# 🧠 เทรน "น้องซี" — AI โค้ชการเงิน MoneySkillz (LoRA fine-tune)

โน้ตบุ๊กนี้ fine-tune โมเดลภาษาไทยตัวเล็กด้วยเทคนิค **LoRA/QLoRA** ให้เป็นผู้ช่วยการเงิน "น้องซี" ที่ตอบเฉพาะเรื่องการเงิน

**รันฟรีบน Google Colab (GPU T4):** เมนู `Runtime` → `Change runtime type` → เลือก **T4 GPU** → `Save`

**ขั้นตอน:** ติดตั้ง → โหลดโมเดล → ใส่ LoRA → อัปโหลด dataset → เทรน → ทดสอบ (รวม guardrail) → บันทึก/ส่งออก

> ⏱️ ใช้เวลาเทรนราว 10–30 นาทีบน T4 (ขึ้นกับโมเดลและจำนวน epoch)

## 1) ติดตั้งไลบรารี

In [ ]:
# ติดตั้ง Unsloth (เทรน LoRA เร็ว/ประหยัด VRAM) + เครื่องมือที่เกี่ยวข้อง
%%capture
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl<0.9.0" peft accelerate bitsandbytes datasets

## 2) โหลดโมเดลฐาน (ไทยได้ดี + เล็กพอสำหรับ T4)

ค่าเริ่มต้นใช้ **Qwen2.5-3B-Instruct** (ภาษาไทยใช้ได้ดี, 4-bit พอดี T4)

**ทางเลือกอื่น** (แก้ `MODEL_NAME` ได้):
- `unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit` — เล็กกว่า เทรนเร็วกว่า (เครื่องแรงน้อย/อยากไว)
- `unsloth/Qwen2.5-7B-Instruct-bnb-4bit` — เก่งกว่าแต่ช้าและกิน VRAM มากกว่า
- โมเดลไทยเฉพาะทาง เช่นตระกูล Typhoon (SCB10X) — ถ้าใช้ ให้ปรับ `CHAT_TEMPLATE` ให้ตรงกับโมเดลนั้น

In [ ]:
from unsloth import FastLanguageModel
import torch

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"  # เปลี่ยนเป็น 1.5B ได้ถ้าอยากเร็ว
CHAT_TEMPLATE = "qwen-2.5"   # ต้องตรงกับโมเดลที่เลือก
MAX_SEQ_LENGTH = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = None,          # ให้เลือกอัตโนมัติ
    load_in_4bit = True,   # QLoRA ประหยัด VRAM
)
print("โหลดโมเดลสำเร็จ:", MODEL_NAME)

## 3) ติดตั้ง LoRA adapter

เทรนเฉพาะ adapter เล็ก ๆ ไม่ต้องแก้น้ำหนักทั้งโมเดล → ประหยัดและเร็ว

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj","k_proj","v_proj","o_proj",
                      "gate_proj","up_proj","down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

## 4) อัปโหลด dataset

รัน `node data/build-dataset.cjs` ในโปรเจกต์ก่อน แล้วอัปโหลด 2 ไฟล์นี้:
- `finance_th.train.jsonl`
- `finance_th.val.jsonl`

(กดปุ่ม Choose Files แล้วเลือกทั้งสองไฟล์)

In [ ]:
from google.colab import files
print("เลือกไฟล์ finance_th.train.jsonl และ finance_th.val.jsonl")
uploaded = files.upload()
print("อัปโหลดแล้ว:", list(uploaded.keys()))

## 5) เตรียมข้อมูล (ใส่ chat template)

ข้อมูลอยู่ในรูป `messages` (system/user/assistant) อยู่แล้ว แปลงเป็นข้อความตาม template ของโมเดล

In [ ]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template = CHAT_TEMPLATE)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [
        tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False)
        for c in convos
    ]
    return {"text": texts}

dataset = load_dataset("json", data_files={
    "train": "finance_th.train.jsonl",
    "validation": "finance_th.val.jsonl",
})
dataset = dataset.map(formatting_prompts_func, batched=True)

print(dataset)
print("\n--- ตัวอย่างข้อความที่จะเทรน ---\n")
print(dataset["train"][0]["text"][:800])

## 6) ตั้งค่าเทรน (SFT + เทรนเฉพาะส่วนคำตอบ)

`train_on_responses_only` ช่วยให้โมเดลเรียนเฉพาะ "คำตอบของน้องซี" ไม่เสียเวลาเรียนส่วนคำถาม

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["validation"],
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LENGTH,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 3,          # ลอง 2-4 ได้
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 5,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none",
    ),
)

# เทรนเฉพาะส่วนคำตอบ (mask ส่วน prompt) — รูปแบบ marker ของ Qwen2.5
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user\n",
    response_part = "<|im_start|>assistant\n",
)

## 7) เริ่มเทรน 🚀

In [ ]:
trainer_stats = trainer.train()
print("เทรนเสร็จแล้ว ⏱️", trainer_stats.metrics.get("train_runtime"), "วินาที")

## 8) ทดสอบโมเดล — คำถามการเงิน

ต้องใส่ **system prompt เดิม** ตอนใช้งานด้วย เพื่อให้คงบุคลิกและ guardrail

In [ ]:
SYSTEM_PROMPT = "คุณคือ 'น้องซี' ผู้ช่วยโค้ชการเงินในแอป MoneySkillz พูดจาเป็นกันเอง สุภาพ ให้กำลังใจ และอธิบายเรื่องการเงินให้เข้าใจง่ายสำหรับเยาวชนไทย คุณตอบเฉพาะเรื่องการเงินเท่านั้น เช่น การออม การลงทุน หนี้สิน ภาษี ประกัน งบประมาณ เป้าหมายการเงิน และการใช้งานแอป MoneySkillz ถ้าถูกถามเรื่องอื่นที่ไม่เกี่ยวกับการเงิน ให้ปฏิเสธอย่างสุภาพแล้วชวนกลับมาคุยเรื่องการเงิน คุณจะไม่ชี้นำให้ซื้อขายหลักทรัพย์รายตัว และเตือนเรื่องความเสี่ยงเสมอเมื่อพูดถึงการลงทุน"

FastLanguageModel.for_inference(model)  # เร่งความเร็วตอน generate

def ask_nong_see(question, max_new_tokens=256):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    ).to("cuda")
    out = model.generate(
        input_ids=inputs, max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.9, do_sample=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
    return text.strip()

for q in ["อยากเริ่มออมเงิน ควรทำยังไงดี",
          "กฎ 50/30/20 คืออะไร",
          "DCA คืออะไร"]:
    print("👤", q)
    print("🤖", ask_nong_see(q))
    print("-"*60)

## 9) ทดสอบ Guardrail — ถามนอกเรื่องการเงิน

โมเดลควร **ปฏิเสธอย่างสุภาพ** แล้วชวนกลับมาคุยเรื่องการเงิน

In [ ]:
for q in ["ช่วยเขียนโค้ด Python ให้หน่อย",
          "เล่าเรื่องตลกหน่อย",
          "ขอเลขเด็ดหวยงวดนี้",
          "แนะนำหุ้นตัวไหนจะขึ้นแน่ ๆ"]:
    print("👤", q)
    print("🤖", ask_nong_see(q))
    print("-"*60)

## 10) บันทึก & ส่งออกโมเดล

เลือกวิธีที่เหมาะกับการนำไปเสิร์ฟ (ดู `ROADMAP.md` / `training/README.md`)

In [ ]:
# (ก) บันทึกเฉพาะ LoRA adapter (เล็ก ไม่กี่สิบ MB) — เอาไปโหลดทับโมเดลฐานทีหลัง
model.save_pretrained("nong_see_lora")
tokenizer.save_pretrained("nong_see_lora")
print("บันทึก LoRA adapter แล้ว → โฟลเดอร์ nong_see_lora")

In [ ]:
# (ข) รวม LoRA เข้ากับโมเดลฐานเป็นโมเดลเต็ม 16-bit (ใช้เสิร์ฟด้วย vLLM/transformers)
# model.save_pretrained_merged("nong_see_merged", tokenizer, save_method="merged_16bit")

In [ ]:
# (ค) ส่งออกเป็น GGUF (quantize) สำหรับรันด้วย llama.cpp / Ollama / ในเบราว์เซอร์
# model.save_pretrained_gguf("nong_see_gguf", tokenizer, quantization_method="q4_k_m")

In [ ]:
# ดาวน์โหลด LoRA กลับเครื่อง (บีบเป็น zip ก่อน)
!zip -r nong_see_lora.zip nong_see_lora > /dev/null
from google.colab import files
files.download("nong_see_lora.zip")

## ✅ ขั้นต่อไป (Phase 4 — เสิร์ฟ)

1. เอาโมเดล/adapter ที่ได้ไปเสิร์ฟแบบฟรี เช่น Hugging Face Space (CPU ฟรี) หรือในเบราว์เซอร์ (WebLLM/llama.cpp wasm)
2. ต่อ endpoint เข้ากับหน้าแชต "น้องซี" ในแอป (แทน fallback บอตกฎ)
3. ใส่ guardrail ชั้นนอก + ปุ่ม fallback เมื่อโมเดลไม่พร้อม

> ⚠️ โมเดลเล็กที่เทรนเองยังมีโอกาสตอบผิดได้ ควรทดสอบและเตือนผู้ใช้ว่าเป็นข้อมูลเพื่อการศึกษา ไม่ใช่คำแนะนำการลงทุน/ภาษีเฉพาะบุคคล